In [6]:
# 1. INSTALL LIBRARY
# !pip install openai pandas openpyxl

import json
import pandas as pd
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
from IPython.display import display

In [7]:
# 2. SETUP & AUTHENTICATION
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("OPENAI")

client = OpenAI(api_key=api_key)

In [8]:
# 3. SYSTEM PROMPT & SCHEMA 

SYSTEM_PROMPT_TEXT = """
You are an expert sociologist and linguist annotating Reddit comments for a study on Generational Stereotypes. 
You will receive a thread context: Subreddit, Post Title, Parent Comment, and Reply.

### 1. THEORETICAL FRAMEWORK (Entman 1993)
When analyzing frames, identify how the text:
- **Defines a Problem:** (e.g., "Gen Z is lazy").
- **Diagnoses a Cause:** (e.g., "Because of TikTok/Participation trophies").
- **Makes a Moral Evaluation:** (e.g., "They are entitled").
- **Suggests a Remedy:** (e.g., "They need to work retail").

### 2. INCLUSION CRITERIA
Mark `include: false` if:
- Content is off-topic (celebrity gossip, gaming bugs).
- The reply is a bot.

### 3. ANNOTATION RULES (Frames)
1. **Work_economy:** Jobs, housing, inflation, retirement, "lazy," workplace interactions (CRITICAL: retail/office conflicts go here).
2. **Values_politics:** Voting, MAGA, woke, environment, religion, social justice.
3. **Family_life_stage:** Parenting, "iPad kids," living at home, mother-in-laws.
4. **Tech_media:** Social media, smartphones, AI, digital literacy.
5. **Competence:** Intelligence, "lead poisoning," life skills (cooking, driving). (Note: If in a job context, use Work_economy).
6. **Health_aging:** Physical health, appearance (wrinkles), healthcare.
7. **Meta_identity:** "This sub," "OK Boomer," complaining about generational labels.

### 4. LINGUISTIC PACKAGING (Parent Only)
- **Generic vs. Specific:** - `Generic`: Claims about the category (e.g., "Boomers are..."). 
    - `Specific`: Anecdotes (e.g., "My dad did X"). 
    - **RULE:** If the speaker mentions a specific relative, it is ALWAYS `Specific`.
- **Label Form:** Note if the label is a bare noun ("The Millennials") or adjective ("Millennial parents").
- **Quantifiers:** Universal (All/Never), Majority (Most), Existential (Some/Few), or None.

### 5. REPLY INTERACTION (Stance & Counters)
- **Stance:** Support, Oppose, Mitigate (Partial agree), Neutral.
- **Counter-Moves:**
    - `Not all`: General exceptions.
    - `Counter-example`: **RULE:** "My [relative] is actually X" is a Counter-example.
    - `Reframing`: Changing the moral judgment (e.g., "Not lazy, just underpaid").
    - `Sarcasm`: Pretending to agree to mock the parent.
"""

# 3. ANNOTATION SCHEMA
ANNOTATION_SCHEMA = {
    "name": "annotation_schema",
    "schema": {
        "type": "object",
        "properties": {
            "include": {"type": "boolean"},
            "reasoning_trace": {
                "type": "string", 
                "description": "Format: PROBLEM: [text], CAUSE: [text], MORAL EVAL: [text]. Then explain stance choice."
            },
            "parent_analysis": {
                "type": "object",
                "properties": {
                    "target_group": {"type": "string", "enum": ["Boomers", "GenX", "Millennials", "GenZ", "Silent", "Mixed"]},
                    "perspective": {"type": "string", "enum": ["Auto", "Hetero", "Mixed", "Unclear"]},
                    "primary_frame": {"type": "string", "enum": ["Work_economy", "Values_politics", "Family_life_stage", "Tech_media", "Competence", "Meta_identity", "Health_aging", "Other"]},
                    "generic_specific": {"type": "string", "enum": ["Generic", "Specific", "Unclear"]},
                    "label_form": {"type": "string", "enum": ["Noun", "Adjective", "Mixed"]},
                    "abstraction": {"type": "string", "enum": ["Trait", "Event", "Mixed"]}, # Moved here
                    "quantifier_type": {"type": "string", "enum": ["Universal", "Majority", "Existential", "None"]},
                    "boosters_present": {"type": "boolean"},
                    "hedges_present": {"type": "boolean"},
                    "valence": {"type": "string", "enum": ["Negative", "Positive", "Mixed", "Neutral"]}
                }
            },
            "reply_analysis": {
                "type": "object",
                "properties": {
                    "stance": {"type": "string", "enum": ["Support", "Mitigate", "Oppose", "Neutral"]},
                    "counter_moves": {
                        "type": "object",
                        "properties": {
                            "not_all": {"type": "boolean"},
                            "counter_example": {"type": "boolean"},
                            "alternative_cause": {"type": "boolean"},
                            "reframing": {"type": "boolean"},
                            "sarcasm": {"type": "boolean"}
                        }
                    }
                }
            }
        },
        "required": ["include", "reasoning_trace"],
        "additionalProperties": False
    }
}

In [9]:
# 4. THE CALL FUNCTION (Context-Rich)
def analyze_row(full_context_string):
    try:
        response = client.chat.completions.create(
            model="gpt-5.2", 
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_TEXT},
                {"role": "user", "content": full_context_string}
            ],
            extra_body={"reasoning_effort": "medium"},
            response_format={
                "type": "json_schema",
                "json_schema": ANNOTATION_SCHEMA 
            }
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"API Error: {e}")
        return None

In [ ]:
import time
import pandas as pd

# 5. RUN TEST BATCH
print("📂 Loading data...")
df = pd.read_json("/kaggle/input/reddit-pairs-3rd/reddit_pairs_3rd.jsonl", lines=True)

# Clean Data and select test batch
df = df[~df['parent_text'].isin(['[deleted]', '[removed]', None])]
full_batch = df.copy() 
full_results = []

print(f"🚀 Starting Run on {len(full_batch)} rows...")
start_time = time.time()

for index, row in full_batch.iterrows():
    if index % 50 == 0 and index > 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / index
        remaining = avg_time * (len(full_batch) - index)
        print(f"⏳ Progress: {index}/{len(full_batch)} rows completed... (Est. {remaining/60:.1f} mins left)")
        
        temp_df = pd.DataFrame(full_results)
        temp_df.to_csv("generational_stereotypes_CHECKPOINT.csv", index=False)

    # Extract Metadata
    sub = str(row.get('subreddit', ''))
    sub_type = "specific" if sub.lower() in ['genz', 'boomersbeingfools', 'millennials', 'genx'] else "general"

    # Context Build
    full_context_input = f"METADATA: r/{sub}\nTITLE: {row.get('post_title')}\nPARENT: {row.get('parent_text')}\nREPLY: {row.get('child_text')}"
    
    try:
        analysis = analyze_row(full_context_input)
        if analysis:
            p = analysis.get("parent_analysis", {})
            r = analysis.get("reply_analysis", {})
            cm = r.get("counter_moves", {})
            
            full_results.append({
                "id": index,
                "subreddit": sub,
                "subreddit_type": sub_type,
                "parent_text": row.get('parent_text'),
                "reply_text": row.get('child_text'),
                "target_group": p.get("target_group"),
                "perspective": p.get("perspective"),
                "frame": p.get("primary_frame"),
                "generic": p.get("generic_specific"),
                "label_form": p.get("label_form"),
                "abstraction": p.get("abstraction"),
                "boosters": p.get("boosters_present"),
                "hedges": p.get("hedges_present"),
                "valence": p.get("valence"),
                "stance": r.get("stance"),
                "reframing": cm.get("reframing"),
                "counter_ex": cm.get("counter_example"),
                "reasoning": analysis.get("reasoning_trace")
            })
    except Exception as e:
        print(f"⚠️ Error at index {index}: {e}")

📂 Loading data...
🚀 Starting Run on 4199 rows...
⏳ Progress: 50/4199 rows completed... (Est. 644.1 mins left)
⏳ Progress: 100/4199 rows completed... (Est. 652.7 mins left)
⏳ Progress: 150/4199 rows completed... (Est. 595.4 mins left)


In [ ]:
# 6. SAVE RESULTS
test_df = pd.DataFrame(full_results)
display(test_df)
test_df.to_csv("generational_stereotypes_full_run.csv", index=False)
print("✅ File saved: generational_stereotypes_full_run.csv")